# Exp 2: Deep SVDD Pretraining Epoch Sensitivity

Reads results from `saves_exp2_svdd_epochs_identity/` and plots normalized return vs SVDD training epoch count.

**Prerequisites:**
```bash
bash train_scripts/exp2_svdd_epochs/train_hopper.sh
```

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gym
import d4rl  # noqa

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
ENV        = 'hopper-medium-v2'
SVDD_EPOCHS = [100, 300, 500, 1000]
N_SEEDS    = 3
SAVE_ROOT  = 'saves_exp2_svdd_epochs_identity'
EVAL_COL   = 'evaluation/Average Returns'

In [ ]:
def get_normalized_score(env_name, raw_return):
    env = gym.make(env_name)
    score = env.get_normalized_score(raw_return) * 100
    env.close()
    return score

def read_final_return(root, env, trial_name):
    pattern = os.path.join(root, '**', trial_name, '**', env, '**', 'progress.csv')
    files = glob.glob(pattern, recursive=True)
    if not files:
        return None
    df = pd.read_csv(files[0])
    if EVAL_COL not in df.columns:
        return None
    return df[EVAL_COL].dropna().iloc[-1]

In [ ]:
rows = []
for svdd_epochs in SVDD_EPOCHS:
    raw_returns = []
    for seed in range(N_SEEDS):
        trial = f'e{svdd_epochs}_s{seed}'
        r = read_final_return(SAVE_ROOT, ENV, trial)
        if r is not None:
            raw_returns.append(r)
        else:
            print(f'[WARN] Missing: epochs={svdd_epochs}, seed={seed}')

    norm = [get_normalized_score(ENV, r) for r in raw_returns] if raw_returns else []
    rows.append({
        'svdd_train_epochs': svdd_epochs,
        'n_seeds':           len(norm),
        'mean':              np.mean(norm) if norm else float('nan'),
        'std':               np.std(norm)  if norm else float('nan'),
    })

df = pd.DataFrame(rows)
df

In [ ]:
epochs = df['svdd_train_epochs'].values
means  = df['mean'].values
stds   = df['std'].values

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, means, marker='o', linewidth=2, color='steelblue', label='Mean (3 seeds)')
ax.fill_between(epochs, means - stds, means + stds,
                alpha=0.25, color='steelblue', label='±1 std')
ax.set_xlabel('SVDD Training Epochs', fontsize=12)
ax.set_ylabel('Normalized Return (%)', fontsize=12)
ax.set_title(f'SVDD Epoch Sensitivity — {ENV}', fontsize=13)
ax.set_xticks(epochs)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

os.makedirs('results', exist_ok=True)
plt.savefig('results/exp2_epoch_sensitivity.png', dpi=150)
plt.show()

In [ ]:
df.to_csv('results/exp2_epoch_sensitivity.csv', index=False)
print('Saved results/exp2_epoch_sensitivity.csv')